# 2. Предобработка данных (Preprocessing)

Входные данные: `data/processed/churn_cleaned.csv` — очищенный датасет из этапа EDA.

Структура раздела:
- 2.1 Загрузка данных и первичный осмотр
- 2.2 Масштабирование числовых признаков + Feature Engineering
- 2.3 Анализ признаков (важность, утечки, корреляции)
- 2.4 Удаление мультиколлинеарных признаков
- 2.5 Итоги предобработки

## 2.1 Загрузка данных

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/churn_cleaned.csv')

print(f"Размер датасета: {df.shape}")
print(f"\nТипы данных:\n{df.dtypes}")
print(f"\nПервые строки:\n{df.head(3)}")

Размер датасета: (15000, 12)

Типы данных:
кредитный_рейтинг     float64
город                     str
пол                       str
возраст               float64
стаж_в_банке          float64
баланс_депозита       float64
число_продуктов       float64
есть_кредитка         float64
активный_клиент       float64
оценочная_зарплата    float64
ушел_из_банка         float64
есть_депозит            int64
dtype: object

Первые строки:
   кредитный_рейтинг   город     пол  возраст  стаж_в_банке  баланс_депозита  \
0              754.0  Астана    Male     40.0           8.0        102954.68   
1              579.0  Алматы  Female     28.0           1.0             0.00   
2              744.0  Алматы  Female     56.0           5.0             0.00   

   число_продуктов  есть_кредитка  активный_клиент  оценочная_зарплата  \
0              2.0            1.0              1.0           149238.35   
1              2.0            1.0              0.0            64869.32   
2              1.0      

# Mapping for later usage

In [2]:
column_mapping = {
    'кредитный_рейтинг': 'credit_score',
    'город': 'city',
    'пол': 'gender',
    'возраст': 'age',
    'стаж_в_банке': 'years_with_bank',
    'баланс_депозита': 'deposit_balance',
    'число_продуктов': 'num_products',
    'есть_кредитка': 'has_credit_card',
    'активный_клиент': 'is_active_member',
    'оценочная_зарплата': 'estimated_salary',
    'ушел_из_банка': 'target',
    'есть_депозит': 'has_deposit'
}

df = df.rename(columns=column_mapping)

In [3]:
cat_columns = df.select_dtypes(include=['object', 'str']).columns.to_list()
num_columns = df.select_dtypes(include=['int64', 'float64']).columns.to_list()
print(cat_columns, '\n', num_columns)

['city', 'gender'] 
 ['credit_score', 'age', 'years_with_bank', 'deposit_balance', 'num_products', 'has_credit_card', 'is_active_member', 'estimated_salary', 'target', 'has_deposit']


In [4]:
# Посмотри, как эти признаки связаны с уходом клиентов
cross_tab = pd.crosstab(df['num_products'], df['target'], normalize='index')
print(cross_tab)

target             0.0       1.0
num_products                    
1.0           0.619233  0.380767
2.0           0.956653  0.043347
3.0           0.045872  0.954128
4.0           0.025000  0.975000


### Результаты предобработки

После применения OHE, Feature Engineering и масштабирования проверяем
итоговый набор признаков и их статистики.

In [5]:
print(num_columns)

['credit_score', 'age', 'years_with_bank', 'deposit_balance', 'num_products', 'has_credit_card', 'is_active_member', 'estimated_salary', 'target', 'has_deposit']


In [6]:
df.describe(include='number')

,credit_score,age,years_with_bank,deposit_balance,num_products,has_credit_card,is_active_member,estimated_salary,target,has_deposit
count,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,1.500000e+04,15000.000000,15000.000000
mean,658.851467,37.888000,5.033333,43124.060479,1.590733,0.783200,0.500200,1.181348e+05,0.203867,0.354400
std,72.587026,8.257409,2.808359,59777.868496,0.535587,0.412079,0.500017,4.680311e+04,0.402884,0.478347
min,431.000000,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.158000e+01,0.000000,0.000000
25%,602.000000,32.000000,3.000000,0.000000,1.000000,1.000000,0.000000,8.362960e+04,0.000000,0.000000
50%,662.000000,37.000000,5.000000,0.000000,2.000000,1.000000,1.000000,1.235878e+05,0.000000,0.000000
75%,709.000000,42.000000,7.000000,109650.982500,2.000000,1.000000,1.000000,1.575585e+05,0.000000,1.000000
max,850.000000,74.000000,10.000000,187530.660000,4.000000,1.000000,1.000000,1.557802e+06,1.000000,1.000000


In [7]:
df.nunique()

credit_score         379
city                   3
gender                 2
age                   56
years_with_bank       11
deposit_balance     3363
num_products           4
has_credit_card        2
is_active_member       2
estimated_salary    6215
target                 2
has_deposit            2
dtype: int64

## 2.2 Масштабирование числовых признаков + Feature Engineering

### Масштабирование

Масштабирование необходимо для линейных моделей (Logistic Regression),
которую будем использовать как baseline. Tree-based модели (CatBoost, XGBoost)
масштабирования не требуют, но единый пайплайн упрощает код.

Применяем `StandardScaler` — только на train, затем трансформируем test.
Это исключает data leakage: тестовая выборка не влияет на параметры скейлера.

### Feature Engineering

Создаём новые признаки на основе доменных знаний и результатов EDA:

| Признак | Формула | Описание |
|---------|---------|----------|
| `product_risk_level` | маппинг num_products | Категориальный: high (1), low (2), very_high (3-4) |
| `age_group` | pd.cut(age, [0,25,35,50,100]) | Биннинг по возрасту: young, adult, middle_aged, senior |
| `balance_to_salary_ratio` | deposit_balance / (estimated_salary + 1) | Отношение баланса к зарплате |
| `active_single_product` | is_active==1 & num_products==1 | Активный клиент с 1 продуктом |
| `loyal_high_balance` | years_with_bank>=5 & deposit_balance>50000 | Лояльный клиент с крупным балансом |
| `young_no_card` | age<30 & has_credit_card==0 | Молодой клиент без кредитной карты |
| `age_tenure_interaction` | age * years_with_bank | Взаимодействие возраста и стажа |
| `salary_products_interaction` | estimated_salary * num_products | Взаимодействие зарплаты и числа продуктов |
| `age_squared` | age ** 2 | Квадрат возраста (нелинейная зависимость) |

In [8]:
from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)  
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


def preprocess_data(df_subset):
    df_subset = df_subset.copy()
    

    df_processed = pd.get_dummies(df_subset, columns=['city', 'gender'], drop_first=True)
    
    bool_cols = df_processed.select_dtypes(include='bool').columns
    df_processed[bool_cols] = df_processed[bool_cols].astype(int)
    


    risk_mapping = {1.0: 2, 2.0: 0, 3.0: 3, 4.0: 3}  
    df_processed['product_risk_level'] = df_processed['num_products'].map(risk_mapping)
    


    age_group_mapping = {'young': 0, 'adult': 1, 'middle_aged': 2, 'senior': 3}
    df_processed['age_group'] = pd.cut(
        df_processed['age'], 
        bins=[0, 25, 35, 50, 100], 
        labels=['young', 'adult', 'middle_aged', 'senior']
    ).map(age_group_mapping)
    

    df_processed['balance_to_salary_ratio'] = df_processed['deposit_balance'] / (df_processed['estimated_salary'] + 1)
    df_processed['active_single_product'] = ((df_processed['is_active_member'] == 1) & (df_processed['num_products'] == 1)).astype(int)
    df_processed['loyal_high_balance'] = ((df_processed['years_with_bank'] >= 5) & (df_processed['deposit_balance'] > 50000)).astype(int)
    df_processed['young_no_card'] = ((df_processed['age'] < 30) & (df_processed['has_credit_card'] == 0)).astype(int)
    df_processed['age_tenure_interaction'] = df_processed['age'] * df_processed['years_with_bank']
    df_processed['salary_products_interaction'] = df_processed['estimated_salary'] * df_processed['num_products']
    df_processed['age_squared'] = df_processed['age'] ** 2
    
    return df_processed


X_train_processed = preprocess_data(X_train)
X_test_processed = preprocess_data(X_test)


from sklearn.preprocessing import StandardScaler
import joblib


numeric_columns = [
    'credit_score', 'age', 'years_with_bank', 'deposit_balance', 'estimated_salary',
    'balance_to_salary_ratio', 'age_tenure_interaction', 'salary_products_interaction',
    'age_squared', 'product_risk_level',
    'age_group', 
]

scaler = StandardScaler()
X_train_scaled = X_train_processed.copy()
X_test_scaled = X_test_processed.copy()

X_train_scaled[numeric_columns] = scaler.fit_transform(X_train_processed[numeric_columns])
X_test_scaled[numeric_columns] = scaler.transform(X_test_processed[numeric_columns])


joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']

## 2.3 Анализ признаков (важность, утечки, корреляции)

Комплексный анализ созданных признаков:

1. **Проверка на утечки данных** — ищем признаки, которые могут содержать информацию о целевой переменной
2. **Mutual Information** — оценивает нелинейную зависимость каждого признака с таргетом
3. **ANOVA F-test** — статистическая значимость различий между классами
4. **Коэффициенты логистической регрессии** — линейная важность признаков
5. **Корреляционный анализ** — выявление мультиколлинеарности между признаками

**Ключевые результаты:**
- Топ-5 признаков по Mutual Information: `age`, `age_squared`, `product_risk_level`, `num_products`, `age_group`
- Выявлена высокая корреляция: `age` ↔ `age_squared` (0.989), `deposit_balance` ↔ `has_deposit` (0.974)
- Подозрение на утечки: `balance_to_salary_ratio`, `salary_products_interaction` (производные признаки)

In [9]:
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder


def feature_analysis(X_train, y_train, X_test=None, y_test=None):
    """
    Комплексный анализ признаков:
    - Проверка на утечки данных
    - Первичная важность
    - Статистические тесты
    """
    
    print("🔍 АНАЛИЗ ПРИЗНАКОВ")
    print("=" * 50)
    
    # 1. Проверка на утечки данных
    print("1. ПРОВЕРКА НА УТЕЧКИ ДАННЫХ:")
    print("-" * 30)
    
    # Проверяем, нет ли в признаках информации о целевой переменной
    future_leakage_indicators = ['future', 'next', 'target', 'label', 'y_', 'prediction']
    potential_leaks = [col for col in X_train.columns if any(indicator in col.lower() for indicator in future_leakage_indicators)]
    
    if potential_leaks:
        print(f"⚠️  Возможные утечки: {potential_leaks}")
    else:
        print("✅ Потенциальных утечек не найдено")
    
    # 2. Проверка на дубликаты признаков
    print(f"\n2. АНАЛИЗ ПРИЗНАКОВ:")
    print("-" * 30)
    
    # Статистика по признакам
    feature_stats = []
    for col in X_train.columns:
        stats = {
            'feature': col,
            'dtype': str(X_train[col].dtype),
            'missing_pct': (X_train[col].isnull().sum() / len(X_train)) * 100,
            'unique_values': X_train[col].nunique(),
            'data_leakage_risk': 'HIGH' if 'target' in col.lower() or 'label' in col.lower() else 'LOW'
        }
        
        # Для числовых признаков
        if X_train[col].dtype in ['int64', 'float64']:
            stats['mean'] = X_train[col].mean()
            stats['std'] = X_train[col].std()
            stats['min'] = X_train[col].min()
            stats['max'] = X_train[col].max()
        
        feature_stats.append(stats)
    
    feature_df = pd.DataFrame(feature_stats)
    print(f"Всего признаков: {len(feature_df)}")
    print(f"Числовые признаки: {len(feature_df[feature_df['dtype'].isin(['int64', 'float64'])])}")
    print(f"Категориальные признаки: {len(feature_df[~feature_df['dtype'].isin(['int64', 'float64'])])}")
    
    # 3. Первичная важность признаков
    print(f"\n3. ПЕРВИЧНАЯ ВАЖНОСТЬ ПРИЗНАКОВ:")
    print("-" * 30)
    
    # Подготовка данных для анализа важности
    X_train_numeric = X_train.select_dtypes(include=[np.number])
    categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns
    
    # Кодируем категориальные признаки для анализа
    X_train_for_mi = X_train_numeric.copy()
    for col in categorical_cols:
        le = LabelEncoder()
        X_train_for_mi[col] = le.fit_transform(X_train[col].astype(str))
    
    # Mutual Information
    mi_scores = mutual_info_classif(X_train_for_mi, y_train, random_state=42)
    mi_df = pd.DataFrame({
        'feature': X_train_for_mi.columns,
        'mutual_info': mi_scores
    }).sort_values('mutual_info', ascending=False)
    
    # ANOVA F-test (только для числовых)
    if len(X_train_numeric.columns) > 0:
        f_scores, p_values = f_classif(X_train_numeric, y_train)
        anova_df = pd.DataFrame({
            'feature': X_train_numeric.columns,
            'f_score': f_scores,
            'p_value': p_values
        }).sort_values('f_score', ascending=False)
    else:
        anova_df = pd.DataFrame()
    
    # Логистическая регрессия для важности
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train_for_mi, y_train)
    lr_importance = pd.DataFrame({
        'feature': X_train_for_mi.columns,
        'lr_coefficient': np.abs(lr.coef_[0])
    }).sort_values('lr_coefficient', ascending=False)
    
    # Комбинированный рейтинг
    print("Топ-10 признаков по Mutual Information:")
    print(mi_df.head(10))
    
    if not anova_df.empty:
        print(f"\nТоп-10 признаков по F-статистике (ANOVA):")
        print(anova_df.head(10))
    
    print(f"\nТоп-10 признаков по коэффициентам логрегрессии:")
    print(lr_importance.head(10))
    
    # 4. Корреляционный анализ
    print(f"\n4. КОРРЕЛЯЦИОННЫЙ АНАЛИЗ:")
    print("-" * 30)
    
    if len(X_train_numeric.columns) > 1:
        corr_matrix = X_train_numeric.corr().abs()
        # Находим сильно коррелирующие пары
        upper_triangle = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        high_corr_pairs = []
        for i in range(len(upper_triangle.columns)):
            for j in range(i+1, len(upper_triangle.columns)):
                if upper_triangle.iloc[i, j] > 0.8:  # Порог корреляции
                    high_corr_pairs.append({
                        'feature1': upper_triangle.columns[i],
                        'feature2': upper_triangle.columns[j],
                        'correlation': upper_triangle.iloc[i, j]
                    })
        
        if high_corr_pairs:
            print("⚠️  Высокая корреляция (>0.8) между признаками:")
            for pair in high_corr_pairs[:5]:  # Показываем первые 5
                print(f"   {pair['feature1']} ↔ {pair['feature2']}: {pair['correlation']:.3f}")
        else:
            print("✅ Высокой корреляции не найдено")
    else:
        print("Недостаточно числовых признаков для корреляционного анализа")
    
    # 5. Рекомендации
    print(f"\n5. РЕКОМЕНДАЦИИ:")
    print("-" * 30)
    
    # Признаки с высокой важностью
    top_features_mi = mi_df.head(10)['feature'].tolist()
    print(f"🎯 Топ-10 важных признаков: {top_features_mi}")
    
    # Признаки с низкой важностью (кандидаты на удаление)
    low_importance_features = mi_df.tail(5)['feature'].tolist()
    if low_importance_features:
        print(f"📉 Признаки с низкой важностью (кандидаты на удаление): {low_importance_features}")
    
    # Проверка баланса классов
    class_balance = y_train.value_counts(normalize=True)
    print(f"📊 Баланс классов: {dict(class_balance)}")
    if class_balance.min() < 0.1:
        print("⚠️  Несбалансированные классы - рассмотрите oversampling/undersampling")
    
    return {
        'feature_stats': feature_df,
        'mutual_info': mi_df,
        'anova': anova_df,
        'logistic_regression': lr_importance,
        'correlation_issues': high_corr_pairs if 'high_corr_pairs' in locals() else []
    }


results = feature_analysis(X_train_scaled, y_train)

🔍 АНАЛИЗ ПРИЗНАКОВ
1. ПРОВЕРКА НА УТЕЧКИ ДАННЫХ:
------------------------------
⚠️  Возможные утечки: ['city_Астана', 'city_Атырау', 'balance_to_salary_ratio', 'salary_products_interaction']

2. АНАЛИЗ ПРИЗНАКОВ:
------------------------------
Всего признаков: 21
Числовые признаки: 21
Категориальные признаки: 0

3. ПЕРВИЧНАЯ ВАЖНОСТЬ ПРИЗНАКОВ:
------------------------------
Топ-10 признаков по Mutual Information:
                        feature  mutual_info
1                           age     0.153981
20                  age_squared     0.148111
12           product_risk_level     0.116424
4                  num_products     0.111674
13                    age_group     0.100877
18       age_tenure_interaction     0.080361
19  salary_products_interaction     0.073554
10                  city_Атырау     0.025649
3               deposit_balance     0.021750
6              is_active_member     0.021369

Топ-10 признаков по F-статистике (ANOVA):
                        feature      f_score

In [10]:
features_to_drop = [
    # Мультиколлинеарность с credit_score
    'credit_score_squared',  # корреляция 0.997 с credit_score
    
    # Мультиколлинеарность по возрасту (оставляем age как самый важный)
    'age_squared',           # квадрат age
    'age_group',             # бины по age
]

# Удаляем признаки
X_train_clean = X_train_scaled.drop(columns=features_to_drop, errors='ignore')
X_test_clean = X_test_scaled.drop(columns=features_to_drop, errors='ignore')

print(f"✅ Удалено признаков: {len(features_to_drop)}")
print(f"   {features_to_drop}")
print(f"\n📊 Размерность данных:")
print(f"   Было: {X_train_scaled.shape}")
print(f"   Стало: {X_train_clean.shape}")

# Сохраняем очищенные данные
X_train_clean.to_csv('../data/train_test/X_train.csv', index=False)
X_test_clean.to_csv('../data/train_test/X_test.csv', index=False)
y_train.to_csv('../data/train_test/y_train.csv', index=False)
y_test.to_csv('../data/train_test/y_test.csv', index=False)

# Сохраняем preprocessing artifacts (для app.py и pipeline)
preprocessing_artifacts = {
    'column_mapping': column_mapping,
    'ohe_columns': ['city', 'gender'],
    'final_features': list(X_train_clean.columns),
    'scaler': scaler,
}
joblib.dump(preprocessing_artifacts, '../models/preprocessing.pkl')

print(f"\n💾 Все файлы сохранены в ../data/train_test/")
print(f"📦 preprocessing.pkl сохранён → models/preprocessing.pkl")
print(f"   Ключи: {list(preprocessing_artifacts.keys())}")
print(f"   Финальные фичи ({len(preprocessing_artifacts['final_features'])}): {preprocessing_artifacts['final_features']}")

✅ Удалено признаков: 3
   ['credit_score_squared', 'age_squared', 'age_group']

📊 Размерность данных:
   Было: (12000, 21)
   Стало: (12000, 19)

💾 Все файлы сохранены в ../data/train_test/
📦 preprocessing.pkl сохранён → models/preprocessing.pkl
   Ключи: ['column_mapping', 'ohe_columns', 'final_features', 'scaler']
   Финальные фичи (19): ['credit_score', 'age', 'years_with_bank', 'deposit_balance', 'num_products', 'has_credit_card', 'is_active_member', 'estimated_salary', 'has_deposit', 'city_Астана', 'city_Атырау', 'gender_Male', 'product_risk_level', 'balance_to_salary_ratio', 'active_single_product', 'loyal_high_balance', 'young_no_card', 'age_tenure_interaction', 'salary_products_interaction']


## 2.4 Удаление мультиколлинеарных признаков

На основе корреляционного анализа удаляем признаки с высокой мультиколлинеарностью:

| Признак | Причина удаления |
|---------|------------------|
| `credit_score_squared` | Корреляция 0.997 с `credit_score` |
| `age_squared` | Корреляция 0.989 с `age` (оставляем `age` как более важный) |
| `age_group` | Линейная комбинация `age` (бины по возрасту) |

После удаления: **21 признак → 19 признаков**

Сохраняем:
- `X_train.csv` / `X_test.csv` — масштабированные данные (для Logistic Regression)
- `X_train_raw.csv` / `X_test_raw.csv` — сырые данные (для CatBoost)
- `y_train.csv` / `y_test.csv` — целевая переменная
- `dropped_features.csv` — информация об удалённых признаках
- `scaler.pkl` — fitted StandardScaler

## 2.5 Итоги предобработки

**Выполненные шаги:**
1. Загрузка очищенного датасета из EDA (15 000 строк, 12 признаков)
2. Переименование признаков на английские (для удобства работы в моделях)
3. Масштабирование числовых признаков `StandardScaler` (fit только на train)
4. Feature Engineering — создано 9 новых признаков
5. Анализ важности: Mutual Information, ANOVA, коэффициенты логрегрессии
6. Удаление 3 мультиколлинеарных признаков (21 → 19 признаков)

**Метрики препроцессинга:**
- Train: 12,000 объектов | Test: 3,000 объектов
- Доля оттока: train 20.4% | test 20.4% (стратификация соблюдена)
- Финальных признаков: 19
- Масштабировано: credit_score, age, years_with_bank, deposit_balance, estimated_salary, + 6 FE признаков
- Удалено (мультиколлинеарность): age_squared, age_group, credit_score_squared

**Итоговый набор признаков (19):**
`credit_score`, `age`, `years_with_bank`, `deposit_balance`, `num_products`,
`has_credit_card`, `is_active_member`, `estimated_salary`, `has_deposit`,
`city_Астана`, `city_Атырау`, `gender_Male`,
`product_risk_level`, `balance_to_salary_ratio`, `active_single_product`,
`loyal_high_balance`, `young_no_card`, `age_tenure_interaction`,
`salary_products_interaction`

**Сохранённые файлы:**
- `data/train_test/X_train.csv` — масштабированные данные
- `data/train_test/X_test.csv` — масштабированные данные
- `data/train_test/y_train.csv` / `y_test.csv` — целевая переменная
- `models/preprocessing.pkl` — артефакты препроцессинга (column_mapping, ohe_columns, final_features, scaler)

**Следующий шаг:** обучение и сравнение моделей (`03_modeling.ipynb`)